In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.patches as mpatches

import MDAnalysis as mda
from MDAnalysis.analysis.dihedrals import Ramachandran
from MDAnalysis.analysis.rms import RMSD, RMSF
from MDAnalysis.analysis.dssp import DSSP, translate

ModuleNotFoundError: No module named 'MDAnalysis'

In [ ]:
WORKDIR = Path.cwd()
input_file = WORKDIR / "1uyd_protein.pdb"
basename = Path(input_file).with_suffix("").name

In [ ]:
from unicodedata import normalize

# Read OpenMM StateDataReporter log

log_file = WORKDIR / "log.txt"
if not log_file.exists():
    for candidate in (
        Path(normalize("NFC", str(log_file))),
        Path(normalize("NFD", str(log_file))),
    ):
        if candidate.exists():
            log_file = candidate
            break
    else:
        matches = list(WORKDIR.rglob("log.txt"))
        if not matches:
            raise FileNotFoundError(f"Could not find log.txt under {WORKDIR}")
        log_file = matches[0]

df = pd.read_csv(log_file, sep='\s+', comment='#', header=0)
df.columns = [c.lstrip("#").strip('"') for c in df.columns]
df

In [ ]:
# Plot directly
plt.figure(figsize=(6, 4))
plt.plot(df["Step"], df["Potential Energy (kJ/mole)"], label="Potential Energy")
plt.xlabel("Step")
plt.ylabel("Energy (kJ/mol)")
plt.legend()
plt.title("Energy Convergence Check")
plt.show()

In [ ]:
output_file = WORKDIR / f"{basename}_output.pdb"
if not output_file.exists():
    for candidate in (
        Path(normalize("NFC", str(output_file))),
        Path(normalize("NFD", str(output_file))),
    ):
        if candidate.exists():
            output_file = candidate
            break
    else:
        matches = list(WORKDIR.rglob(f"{basename}_output.pdb"))
        if not matches:
            raise FileNotFoundError(f"Could not find {basename}_output.pdb under {WORKDIR}")
        output_file = matches[0]

u = mda.Universe(str(output_file))
protein = u.select_atoms("protein")
bb = protein.select_atoms("backbone")
calphas = protein.select_atoms("name CA")
protein = u.select_atoms("protein")
bb = protein.select_atoms("backbone")
calphas = protein.select_atoms("name CA")

In [ ]:
frames = [0, len(u.trajectory) - 1]

fig, ax = plt.subplots()

for f in frames:
    u.trajectory[f]
    protein = u.select_atoms("protein")

    rama = Ramachandran(protein).run()
    angles = rama.results.angles[f]

    phi = ((angles[:, 0] + 180) % 360) - 180
    psi = ((angles[:, 1] + 180) % 360) - 180

    ax.scatter(phi, psi, s=10, label=f"frame {f}", alpha=0.5)

ax.set_xlim(-180, 180)
ax.set_ylim(-180, 180)
ax.set_xlabel("phi (degrees)")
ax.set_ylabel("psi (degrees)")
ax.legend()

plt.show()

In [ ]:
# RMSD
# https://docs.mdanalysis.org/1.1.1/documentation_pages/analysis/rms.html
rmsd = RMSD(u, u, select="backbone").run()

plt.plot(rmsd.results.rmsd[:, 1], rmsd.results.rmsd[:, 2])
plt.xlabel("Frame")
plt.ylabel("RMSD (Å)")
plt.show()

In [ ]:
rmsf = RMSF(calphas).run()

plt.plot(rmsf.results.rmsf)
plt.xlabel("Residue index")
plt.ylabel("RMSF (Å)")
plt.show()

In [ ]:
#https://docs.mdanalysis.org/dev/documentation_pages/analysis/dssp.html
dssp = DSSP(u).run()
ss = dssp.results.dssp[0]

mapping = {
    "H": 1,
    "E": 2,
    "-": 0
}

colors = {
    1: "tab:red",
    2: "tab:blue",
    0: "white"
}

x = np.arange(len(ss))
y = np.ones(len(ss))

color_values = [colors[mapping.get(s, 0)] for s in ss]

legend_elements = [
    mpatches.Patch(
        color=colors[mapping[k]],
        label={"H": "Alpha Helix", "E": "Beta sheet", "-": "Coil"}[k]
    )
    for k in mapping
]

plt.figure(figsize=(12, 2))
plt.bar(x, y, color=color_values, width=1.0)
plt.yticks([])
plt.legend(handles=legend_elements,  bbox_to_anchor=(1.07, 1), frameon=False)
plt.xlabel("Residue")
plt.xlim(0, len(ss))
plt.tight_layout()
plt.show()